<a href="https://colab.research.google.com/github/cermegno/langgraph-learning/blob/main/03-nvidia-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundation RAG pipeline with Nvidia endpoints
Graph for RAG using Nvidia endpoints and in-memory vector store.

In [ ]:
!pip install langgraph langchain langchain-nvidia-ai-endpoints langchain-text-splitters langchain-core langchain-community --quiet

In [ ]:
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from google.colab import userdata
apikey = userdata.get('apikey')

In [ ]:
llm = ChatNVIDIA(
    base_url = "https://integrate.api.nvidia.com/v1",
    #api_key = apikey,       #This is the OpenAI format
    nvidia_api_key = apikey, #The Langchain Nvidia endpoing requires api with this name
    model="meta/llama-3.1-8b-instruct"
)

embed_model = NVIDIAEmbeddings(model="nvidia/nv-embed-v1")

## Get data for the knowledgebase

In [ ]:
import os
folder_name = "data"
os.makedirs(folder_name, exist_ok=True)
!wget -O data/notes.txt https://raw.github.com/StrategicalIT/ProjectHotdog/main/data/notes.txt

In [ ]:
with open("/content/data/notes.txt", "r") as f:
  docs = f.read()
  f.close()

## Create the Retriever

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

chunks = text_splitter.create_documents([docs])
print(f"Created {len(chunks)} chunks")

vectorstore = InMemoryVectorStore.from_documents(
        documents=chunks,
        embedding=embed_model,
    )
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


## LLM definition

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("placeholder", "{messages}"),
    ]
)

chain = prompt | llm

def call_nvidia_llm(state: State) -> State:
  """Calls Nvidia LLM and updates state"""
  response = chain.invoke({"messages": state['messages']})
  return {"messages": [response]}
#print(call_nvidia_llm({"messages": [HumanMessage(content="Hi, I am Tim")]}))

## Create the graph

In [ ]:
class State(TypedDict):
  # We need an additional list attribute to store the retrieved context
  messages: Annotated[list[BaseMessage], add_messages]
  context: List[Document]

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))